# Diabetic kidney disease risk modelling: worked solution

Worked solution for the exercise in the lesson **[Diabetic kidney disease risk modelling](https://ramkumarmd.com/medical-ai/applied-diabetes/diabetic-kidney-disease-risk)** (*Applied case studies: diabetes & metabolic health* track, Medical AI course by Dr. Ram Kumar G). Try the [exercise notebook](03-diabetic-kidney-disease-risk.ipynb) first. Other answers can be just as good: the written parts show one reasonable answer, not the only one.

**In short:** In diabetes, the risk of kidney failure depends above all on two tests: eGFR, which measures kidney function, and urine albumin-to-creatinine ratio (ACR), which measures kidney damage. KDIGO combines them into a colour-coded 'heat map' of risk categories, and validated equations such as the Kidney Failure Risk Equation turn them into an individual 2- or 5-year risk. On synthetic data, a continuous model using the same tests ranked patients slightly better than the categories, and sparse categories gave unreliable estimates.

> All data in this notebook are synthetic or openly licensed. Never use real patient data here, and never paste identifiable patient information into public AI tools. Educational use only: the code and models are not medical devices.

## Setup

In [ ]:
%pip install -q lifelines

## Worked example

These cells reproduce the lesson's code. Lines starting with `#` after the code show the output you should see (numbers can differ slightly with other library versions).

### Rebuilding the heat map

A synthetic cohort of 20,000 adults with diabetes and the observed 5-year kidney failure rate (%) by category.

In [ ]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(12)
n = 20000
df = pd.DataFrame({
    "age": rng.normal(62, 11, n).clip(30, 90).round(),
    "egfr": rng.gamma(9, 7.5, n).clip(8, 130).round(),          # mL/min/1.73m2
    "acr": np.exp(rng.normal(3.2, 1.6, n)).clip(2, 3000).round(), # urine albumin-to-creatinine, mg/g
})
# Synthetic 5-year risk of kidney failure: higher with lower eGFR and higher albuminuria
logit = -2.6 - 0.075 * (df.egfr - 45) + 0.55 * (np.log(df.acr) - np.log(30)) - 0.01 * (df.age - 62)
df["kidney_failure_5y"] = rng.binomial(1, 1 / (1 + np.exp(-logit)))

df["G"] = pd.cut(df.egfr, [0, 15, 30, 45, 60, 90, 999], right=False,
                 labels=["G5", "G4", "G3b", "G3a", "G2", "G1"])
df["A"] = pd.cut(df.acr, [0, 30, 300, 1e9], right=False, labels=["A1", "A2", "A3"])
heat = df.pivot_table(index="G", columns="A", values="kidney_failure_5y", aggfunc="mean", observed=False)
print((heat.loc[["G1", "G2", "G3a", "G3b", "G4", "G5"]] * 100).round(1).to_string())

# A      A1    A2    A3
# G
# G1    0.0   0.2   0.5
# G2    0.3   2.1   4.4
# G3a   1.7   6.7  18.2
# G3b   7.1  16.5  42.3
# G4   19.7  39.4  52.2
# G5    0.0  50.0   NaN

### Sparse categories

Continuing from the previous block: how many people fall in each cell?

In [ ]:
counts = df.pivot_table(index="G", columns="A", values="kidney_failure_5y", aggfunc="size", observed=False)
print(counts.loc[["G1", "G2", "G3a", "G3b", "G4", "G5"]].to_string())

# A      A1    A2   A3
# G
# G1   1710  1247  192
# G2   4916  3457  477
# G3a  2708  1992  330
# G3b  1456  1014  137
# G4    193   142   23
# G5      4     2    0

### Categories versus a continuous model

Continuing from the previous blocks: heat-map cell risk against a logistic model of eGFR, log ACR and age.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split

train, test = train_test_split(df, test_size=0.3, random_state=0, stratify=df.kidney_failure_5y)
# 1. Heat-map cell: use the observed risk of each G x A cell in the training data
cell_risk = train.groupby(["G", "A"], observed=False).kidney_failure_5y.mean()
p_cell = test.set_index(["G", "A"]).index.map(cell_risk).to_numpy(dtype=float)
# 2. Continuous model using the same information
features = lambda d: np.column_stack([d.egfr, np.log(d.acr), d.age])
model = LogisticRegression(max_iter=1000).fit(features(train), train.kidney_failure_5y)
p_model = model.predict_proba(features(test))[:, 1]
print(f"KDIGO heat-map categories: AUROC {roc_auc_score(test.kidney_failure_5y, p_cell):.3f}")
print(f"Continuous eGFR, log ACR, age: AUROC {roc_auc_score(test.kidney_failure_5y, p_model):.3f}")

# KDIGO heat-map categories: AUROC 0.849
# Continuous eGFR, log ACR, age: AUROC 0.867

## Exercise

**Goal:** Model kidney risk in diabetes with categories, continuous predictors and survival methods.

**Data:** Synthetic cohort generated in the notebook (no real patient data) (Generated in the notebook)

### Step 1

Reproduce the heat map and cell counts, and add bootstrap confidence intervals to each cell's risk.

In [ ]:
ORDER = ["G1", "G2", "G3a", "G3b", "G4", "G5"]
rng_b = np.random.default_rng(0)
boot = []
for _ in range(300):
    s = df.sample(len(df), replace=True, random_state=int(rng_b.integers(1e9)))
    boot.append(s.pivot_table(index="G", columns="A", values="kidney_failure_5y", aggfunc="mean", observed=False))
stack = np.stack([b.reindex(index=ORDER, columns=["A1", "A2", "A3"]).to_numpy(dtype=float) for b in boot])
lo, hi = np.nanpercentile(stack, 2.5, axis=0), np.nanpercentile(stack, 97.5, axis=0)
cells = heat.loc[ORDER]
table = pd.DataFrame(
    [[f"{100 * cells.iloc[i, j]:.1f} ({100 * lo[i, j]:.1f}-{100 * hi[i, j]:.1f}), n={counts.loc[ORDER].iloc[i, j]}"
      if counts.loc[ORDER].iloc[i, j] > 0 else "no patients" for j in range(3)] for i in range(len(ORDER))],
    index=ORDER, columns=["A1", "A2", "A3"])
print("5-year risk % (95% bootstrap interval), number of patients")
print(table.to_string())

Cells with thousands of patients have narrow intervals; the G4 and G5 cells, with a few dozen patients or fewer, are
so uncertain that their risks barely mean anything (G5 A2 is based on 2 patients). A heat map that shows only the
point estimate hides this.

### Step 2

Add follow-up time and censoring, and refit the model as a Cox model from the survival analysis lesson.

In [ ]:
from lifelines import CoxPHFitter

rng_t = np.random.default_rng(13)
lp = -0.075 * (df.egfr - 45) + 0.55 * (np.log(df.acr) - np.log(30)) - 0.01 * (df.age - 62)
event_time = rng_t.exponential(1 / (0.012 * np.exp(lp)))          # years to kidney failure
censor_time = rng_t.uniform(1, 8, len(df))                         # end of follow-up varies
surv = pd.DataFrame({"egfr": df.egfr, "log_acr": np.log(df.acr), "age": df.age,
                     "years": np.minimum(event_time, censor_time),
                     "kidney_failure": (event_time <= censor_time).astype(int)})
print(f"{surv.kidney_failure.sum()} events; {1 - surv.kidney_failure.mean():.0%} censored")
cox = CoxPHFitter().fit(surv, duration_col="years", event_col="kidney_failure")
print(cox.summary[["coef", "exp(coef)", "exp(coef) lower 95%", "exp(coef) upper 95%"]].round(3))
print("True coefficients: egfr -0.075, log_acr 0.55, age -0.01")

With time and censoring, the Cox model uses every patient for the time they were followed, instead of treating
people censored early as event-free. Its coefficients recover the values used to simulate the data.

### Step 3

Add a competing risk of death and compare kidney failure risk estimates with and without it.

In [ ]:
from lifelines import AalenJohansenFitter, KaplanMeierFitter

death_time = rng_t.exponential(1 / (0.02 * np.exp(0.07 * (df.age - 62))))   # deaths from other causes, rising with age
first = np.minimum.reduce([event_time, death_time, censor_time])
event = np.select([first == event_time, first == death_time], [1, 2], 0)
km = KaplanMeierFitter().fit(first, event == 1)
aj = AalenJohansenFitter(seed=0).fit(first, event, event_of_interest=1)
print(f"Deaths before kidney failure: {np.sum(event == 2)}; kidney failures: {np.sum(event == 1)}")
print(f"5-year kidney failure risk, 1 - Kaplan-Meier (deaths censored): {1 - km.predict(5):.1%}")
print(f"5-year kidney failure risk, Aalen-Johansen (death as competing): {aj.cumulative_density_.loc[:5].iloc[-1, 0]:.1%}")
old = df.age >= 75
for label, m in [("age 75+", old), ("under 75", ~old)]:
    km_m = KaplanMeierFitter().fit(first[m], event[m] == 1)
    aj_m = AalenJohansenFitter(seed=0).fit(first[m], event[m], event_of_interest=1)
    print(f"  {label}: 1-KM {1 - km_m.predict(5):.1%}, Aalen-Johansen {aj_m.cumulative_density_.loc[:5].iloc[-1, 0]:.1%}")

Ignoring death overstates kidney failure risk, and most of all, relatively, in older patients, who are more likely to
die first (here 2.8% against 2.3% at age 75 and over, about a fifth too high; 3.9% against 3.7% under 75). The
absolute gaps are small because kidney failure is rare in this cohort; in groups with high kidney risk and high
mortality the difference is much larger. That
is why kidney-risk tools for older adults account for competing mortality: a high "kidney failure risk" from 1 - KM
can lead to unnecessary preparation for dialysis in someone more likely to die of other causes.

### Step 4

Check calibration of the continuous model by decile of predicted risk.

In [ ]:
test = test.assign(pred=p_model, decile=pd.qcut(p_model, 10, labels=False, duplicates="drop"))
cal = test.groupby("decile").agg(predicted=("pred", "mean"), observed=("kidney_failure_5y", "mean"), n=("pred", "size"))
print((cal[["predicted", "observed"]] * 100).round(1).assign(n=cal.n).to_string())

The continuous logistic model was fitted to data generated by a logistic model, so it is well calibrated by decile.
Most of the population sits in the lowest deciles with risks near zero; calibration in the top decile, where
decisions are made, matters most and should be reported with a confidence interval.

### Step 5

Write a short explanation of a patient's 5-year risk that a person with diabetes could understand.

Example explanation for a patient (for a synthetic patient with an estimated 5-year risk of 12%):

> "Your kidney tests show that your kidneys are working at about half the usual rate, and there is extra protein in
> your urine. Using these results and your age, we estimate that out of 100 people like you, about 12 might need
> dialysis or a kidney transplant within the next 5 years, and about 88 would not.
>
> This is an estimate for people with results like yours, not a prediction of what will happen to you. Your risk can
> come down: keeping blood pressure and blood sugar under control, and medicines that protect the kidneys, all lower
> it. We will check your kidney tests again in 3 to 6 months and talk about what the results mean."

Choices behind it: natural frequencies ("12 out of 100") rather than percentages, the outcome described in plain words,
the uncertainty stated, and what the person can do about it.

---

Lesson: [https://ramkumarmd.com/medical-ai/applied-diabetes/diabetic-kidney-disease-risk](https://ramkumarmd.com/medical-ai/applied-diabetes/diabetic-kidney-disease-risk). Code in this notebook is released under the MIT licence; the text under CC BY 4.0. See the repository README for details.